Q1. 파일 열고 크기 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day01_bottling.csv")

print("shape (행, 열):", df.shape)
print("행 수:", df.shape[0])
print("열 수:", df.shape[1])

df.head()

shape (행, 열): (4800, 13)
행 수: 4800
열 수: 13


,lot_id,produced_at,plant_code,line_id,shift,product,fill_error,cap_torque,seal_temp,line_speed,ambient_temp,ambient_humidity,result
0,L00001,2026-04-01 06:45,P-SEOUL-01,F-3,주간,1L,0.06,2.22,179.3,393,18.8,61.8,합격
1,L00002,2026-04-01 06:54,P-SEOUL-01,F-3,주간,1L,0.20,2.27,181.8,408,27.5,51.9,합격
2,L00003,2026-04-01 07:23,P-SEOUL-01,F-1,주간,1L,1.85,2.17,180.4,390,24.2,53.3,합격
3,L00004,2026-04-01 08:26,P-SEOUL-01,F-1,주간,2L,0.66,1.88,178.7,414,23.9,46.5,합격
4,L00005,2026-04-01 08:27,P-SEOUL-01,F-1,주간,2L,-0.17,2.22,181.0,403,21.7,53.7,합격


Q2. 검사 결과 살펴보기

In [2]:
df["합격"].value_counts()
df["재검"].value_counts()

KeyError: '합격'

In [4]:
결과별_건수 = df["result"].value_counts()
print(결과별_건수)

합격_비율 = (df["result"] == "합격").mean() * 100
print("합격 비율:", round(합격_비율, 2), "%")
재검_비율 = (df["result"] == "재검").mean() * 100
print("재검 비율:", round(재검_비율, 2), "%")

result
합격    4554
재검     246
Name: count, dtype: int64
합격 비율: 94.88 %
재검 비율: 5.12 %


Q3. 제품 규격별로 몇 건씩인가

In [10]:
print(df["product"].unique())

print(df["product"].value_counts())

<StringArray>
['1L', '2L', '500mL']
Length: 3, dtype: str
product
500mL    2428
1L       1397
2L        975
Name: count, dtype: int64


Q4. 유독 뜨겁게 밀봉한 묶음 찾기

In [16]:
df.sort_values("seal_temp", ascending=False)[["lot_id", "line_id", "seal_temp", "result"]].head(5)

,lot_id,line_id,seal_temp,result
4554,L04555,F-2,209.5,재검
3946,L03947,F-2,207.2,재검
4031,L04032,F-2,207.0,재검
4703,L04704,F-2,205.9,재검
4383,L04384,F-2,205.5,재검


Q5. 라인별로 재검률이 다른가

In [21]:
df["재검여부"] = (df["result"] == "재검")
df.groupby("line_id")["재검여부"].mean() * 100


line_id
F-1    4.652378
F-2    5.287897
F-3    5.654008
Name: 재검여부, dtype: float64

Q6. 비어 있는 칸 찾기

In [20]:
print(df.isna().sum())
print("빈칸 있는 열 개수:",(df.isna().sum()>0).sum())

lot_id                0
produced_at           0
plant_code            0
line_id               0
shift                 0
product               0
fill_error            0
cap_torque          139
seal_temp             0
line_speed            0
ambient_temp          0
ambient_humidity     74
result                0
dtype: int64
빈칸 있는 열 개수: 2


Q7. 캡 토크로 거르면 몇 건이 빠질까

In [27]:
print("전체:", len(df))
print("2.0 초과:", len(df[df["cap_torque"] > 2.0]))
print("2.0 이하:", len(df[df["cap_torque"] <= 2.0]))
print("합:", len(df[df["cap_torque"] > 2.0]) + len(df[df["cap_torque"] <= 2.0]))

전체: 4800
2.0 초과: 3340
2.0 이하: 1321
합: 4661


Q8. 교대조에 따라 다른가

In [28]:
캡토크_평균 = df.groupby("shift")["cap_torque"].mean()
print(캡토크_평균)


shift
야간    2.039520
오후    2.084021
주간    2.090178
Name: cap_torque, dtype: float64


## Q9. 밀봉 온도에 관리선 긋기 ⭐ (이 미션의 핵심)

밀봉 온도의 평균과 표준편차를 구하고, **평균 ± 표준편차 3배**로 위아래 관리선을 계산하세요.

그다음 아래를 차례로 확인하세요.

1. 관리선을 **벗어난 묶음이 몇 건**인가
2. 그중 **실제 재검은 몇 건**인가 → 나머지는 무엇인가
3. 전체 재검 246건 중 **관리선에 걸린 건 몇 건**인가 → 못 잡은 건 몇 건인가

**결과물**: 관리선 두 개 + 위 세 숫자 + **아래 두 질문에 대한 답 한 줄씩**

In [29]:
평균 = df["seal_temp"].mean()
표준편차 = df["seal_temp"].std()

위선 = 평균 + 3 * 표준편차
아래선 = 평균 - 3 * 표준편차

print("평균:", round(평균, 2))
print("관리 상한:", round(위선, 2))
print("관리 하한:", round(아래선, 2))

# 위로 넘었거나 아래로 넘은 묶음만 고른다
벗어남 = df[(df["seal_temp"] > 위선) | (df["seal_temp"] < 아래선)]

print()
print("관리선 벗어난 건수:", len(벗어남))
print(벗어남["result"].value_counts())
print()
print("전체 재검:", (df["result"] == "재검").sum())

평균: 180.35
관리 상한: 188.3
관리 하한: 172.39

관리선 벗어난 건수: 52
result
합격    35
재검    17
Name: count, dtype: int64

전체 재검: 246


## Q10. 열마다 한 줄로 요약하기

숫자로 된 열들에 대해 **열 하나가 한 줄이 되는 진단표**를 만드세요. 각 줄에는 아래가 들어가야 합니다.

- 열 이름 · 빈칸 비율 · 서로 다른 값의 개수 · 표준편차 · 최솟값 · 최댓값

그리고 **표를 보고 판단**하세요. 이 데이터에서 **분석에 쓸모없는 열이 하나 있습니다.** 어느 열이고 왜 그런지 적으세요.

**결과물**: 진단표 + 쓸모없는 열 하나와 그 이유 한 줄

In [30]:
숫자열 = ["fill_error", "cap_torque", "seal_temp",
          "line_speed", "ambient_temp", "ambient_humidity"]

진단표 = pd.DataFrame({
    "빈칸비율(%)": (df[숫자열].isna().sum() / len(df) * 100).round(2),
    "값종류수": df[숫자열].nunique(),
    "표준편차": df[숫자열].std().round(3),
    "최솟값": df[숫자열].min(),
    "최댓값": df[숫자열].max(),
})

진단표

,빈칸비율(%),값종류수,표준편차,최솟값,최댓값
fill_error,0.00,570,1.093,-5.49,5.50
cap_torque,2.90,124,0.165,1.21,2.58
seal_temp,0.00,183,2.652,173.40,209.50
line_speed,0.00,81,12.048,351.00,442.00
ambient_temp,0.00,193,3.235,14.30,36.30
ambient_humidity,1.54,423,7.918,22.40,82.50
